# 12.14 · LLM 智能体 / LLM Agents

> **课程定位 / Where this fits**
> 第 14 课，**Part 12**。让 LLM 从"会回答"升级为"**会做事**"。
> Lesson 14, **Part 12**. Upgrading an LLM from "answers" to "**gets things done**."
>
> 单独的 LLM 只能**凭训练记忆生成文本**——不会算精确的数、查不到实时信息、不能操作外部世界。**LLM 智能体(Agent)** 把 LLM 当作"大脑"，给它一组**工具(tools)**(搜索、计算器、数据库、代码执行、API…)和一个**循环**：LLM **推理**该做什么 → **调用工具** → 看**结果** → 继续推理……直到完成任务。这就是 ReAct(12.11)思想的工程化，是 AutoGPT、Cursor、各种"AI 助手"的核心。本课**从零搭一个工具调用 Agent 循环**。
> An LLM alone only **generates text from training memory** — can't do exact math, fetch live info, or act on the world. An **LLM agent** uses the LLM as a "brain" with a set of **tools** (search, calculator, database, code execution, APIs…) and a **loop**: the LLM **reasons** about what to do → **calls a tool** → sees the **result** → reasons again… until the task is done. This is ReAct (12.11) engineered — the core of AutoGPT, Cursor, and "AI assistants." We **build a tool-using agent loop from scratch**.
>
> 💼 **实战/面试视角**："Agent 循环/ReAct / 工具调用(function calling) / 规划与记忆 / 多智能体 / 风险" 是 LLM 应用前沿。
> 💼 **Practical/interview angle:** "agent loop/ReAct / tool use (function calling) / planning & memory / multi-agent / risks" — the LLM-application frontier.

> 📐 **符号约定 / Notation**
> - 工具(tool) —— Agent 可调用的函数(算/查/执行) / a callable function the agent can use
> - Thought/Action/Observation —— ReAct 循环的三要素 / the ReAct loop's three parts

> 💡 **面试相关 / Interview-relevant**
> - "Agent = LLM + 工具 + 循环"（出镜率 ★★★★）
> - "ReAct 循环(推理-行动-观察)"（★★★★★）
> - "function calling 怎么工作"（★★★★）
> - "Agent 的规划/记忆/反思"（★★★）
> - "Agent 的风险(失控/无限循环/安全)"（★★★★）

---

## 学习目标 / Learning Objectives
1. 理解 Agent = LLM 大脑 + 工具 + 循环。
   Understand agent = LLM brain + tools + loop.
2. 掌握 **ReAct 循环**(推理-行动-观察)。
   Master the ReAct loop (reason-act-observe).
3. **从零搭一个工具调用 Agent**, 完成多步任务。
   Build a tool-using agent from scratch for multi-step tasks.
4. 了解规划/记忆/多智能体与风险。
   Know planning/memory/multi-agent and risks.

## 目录 / TOC
1. [从聊天机器人到 Agent ⭐](#1)
2. [ReAct 循环与工具 ⭐](#2)
3. [从零搭一个 Agent ⭐](#3)
4. [规划/记忆/多智能体/风险 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 从聊天机器人到 Agent ⭐ / From Chatbot to Agent

普通 LLM 聊天：你问 → 它**凭记忆生成**一段文字。局限很明显：
A plain LLM chat: you ask → it **generates from memory**. Clear limits:
- **算不准**：问"34857 × 12.5 = ?"它常算错(语言模型不是计算器)。
  **Bad at math:** "34857 × 12.5 = ?" often wrong (LMs aren't calculators).
- **不知道实时/私有信息**：今天天气、你数据库里的数据。
  **No live/private info:** today's weather, your database.
- **不能操作世界**：发邮件、下单、改文件。
  **Can't act:** send email, place orders, edit files.

**Agent 的解法**：给 LLM 配上**工具**和一个**自主循环**。LLM 不再直接答，而是**决定调用哪个工具、传什么参数**，工具返回真实结果，LLM 据此继续。于是 LLM 的"智能"(理解、推理、规划) + 工具的"能力"(精确计算、实时数据、执行动作)结合起来——**会思考、也会做事**。
**The agent solution:** equip the LLM with **tools** and an **autonomous loop**. Instead of answering directly, the LLM **decides which tool to call with what arguments**; the tool returns a real result; the LLM continues. So the LLM's "intelligence" (understanding, reasoning, planning) + tools' "capabilities" (exact compute, live data, actions) combine — **thinks *and* does**.


<a id="2"></a>
## 2. ReAct 循环与工具 ⭐ / The ReAct Loop & Tools

Agent 的经典控制流是 **ReAct(Reason + Act)** 循环(12.11 见过)：
The classic agent control flow is the **ReAct (Reason + Act)** loop (seen in 12.11):
```
循环:
  Thought(思考):     LLM 想"现在该干什么"
  Action(行动):      LLM 输出要调用的工具 + 参数
  Observation(观察): 系统执行工具, 把结果返回给 LLM
  ... (重复, 直到 LLM 认为可以回答)
  Answer(回答):      LLM 给出最终答案
```
**工具调用(function calling)**：现代 LLM 被训练成能输出**结构化的工具调用**(如 JSON `{"tool":"calculator","args":"34857*12.5"}`)。系统解析它、真正执行函数、把结果作为下一轮的 Observation 喂回去。
**Function calling:** modern LLMs are trained to emit **structured tool calls** (e.g. JSON `{"tool":"calculator","args":"34857*12.5"}`). The system parses it, actually runs the function, and feeds the result back as the next Observation.

> 本机没有真实 LLM, 所以下面用一个**基于规则的"大脑"**来代替 LLM 做决策(决定调用哪个工具)——**重点是展示 Agent 的循环架构**(工具注册表、动作解析、观察反馈、终止条件), 这部分和真实 Agent 完全一样; 真实系统里只是把"规则大脑"换成"LLM 大脑"。
> No real LLM here, so we use a **rule-based "brain"** to stand in for the LLM's decisions — **the point is the agent loop architecture** (tool registry, action parsing, observation feedback, termination), which is identical to real agents; a real system just swaps the rule-brain for an LLM-brain.


<a id="3"></a>
## 3. 从零搭一个 Agent ⭐ / Build an Agent From Scratch

我们搭一个能**多步使用工具**的 Agent：工具有**计算器**和**知识查询**。看它如何把一个需要"先查、再算"的问题拆成多步、调用工具、最后给出答案。
We build an agent that **uses tools over multiple steps**: tools are a **calculator** and a **knowledge lookup**. Watch it decompose a "look up, then compute" question into steps, call tools, and answer.


In [ ]:
import re

# --- 工具注册表: Agent 可用的工具 / tool registry ---
KNOWLEDGE = {"france_population": 67, "japan_population": 125, "usa_population": 331}  # 单位:百万 / millions
def calculator(expr):
    """安全地计算一个算术表达式 / safely evaluate an arithmetic expression."""
    if not re.fullmatch(r"[\d\.\s\+\-\*\/\(\)]+", expr): return "错误: 非法表达式"
    try: return str(eval(expr))
    except Exception as e: return f"错误: {e}"
def lookup(key):
    """从知识库查一个事实 / look up a fact."""
    return str(KNOWLEDGE.get(key, "未找到"))
TOOLS = {"calculator": calculator, "lookup": lookup}

# --- "大脑": 决定下一步动作 (真实系统里这是 LLM; 这里用规则模拟) / the "brain" (rule-based stand-in for an LLM) ---
def brain(question, scratchpad):
    """看问题+已有观察, 决定: 调用工具(tool,args) 还是 给出最终答案(answer) / decide next action."""
    q = question.lower()
    done = {step["tool"] for step in scratchpad}
    if "population" in q and "lookup" not in done:                      # 还没查人口 → 先查 / look up first
        for country in ["france", "japan", "usa"]:
            if country in q: return ("lookup", f"{country}_population")
    if "lookup" in done and ("calculator" not in done) and any(w in q for w in ["times","divided","plus","double","half","sum"]):
        val = scratchpad[-1]["observation"]                            # 用查到的值做算术 / compute with the looked-up value
        if "double" in q or "times 2" in q: return ("calculator", f"{val}*2")
        if "half" in q or "divided by 2" in q: return ("calculator", f"{val}/2")
    return ("answer", scratchpad[-1]["observation"] if scratchpad else "我需要更多信息")

def run_agent(question, max_steps=5):
    print(f"问题: {question}\n" + "-"*50)
    scratchpad = []
    for step in range(max_steps):
        action, arg = brain(question, scratchpad)                      # Thought + Action(大脑决定) / decide
        if action == "answer":
            print(f"  💡 最终答案: {arg}"); return arg
        obs = TOOLS[action](arg)                                       # 执行工具 → Observation / run tool
        print(f"  🤔 Thought: 需要用 {action}")
        print(f"  🔧 Action: {action}('{arg}')")
        print(f"  👀 Observation: {obs}")
        scratchpad.append({"tool": action, "arg": arg, "observation": obs})
    return "(达到最大步数)"

run_agent("What is the population of Japan, doubled?")
print()
run_agent("What is the population of France?")
print("\nAgent 循环: 大脑决定动作→执行工具→观察结果→再决定…直到能回答; 多步任务被自动拆解")


<a id="4"></a>
## 4. 规划/记忆/多智能体/风险 + 小结 ⭐ / Planning, Memory, Multi-Agent & Risks

真实 Agent 在这个基本循环上还有很多扩展(面试可展开)：
Real agents extend this basic loop (good to elaborate in interviews):
- **规划(planning)**：复杂任务先**分解成子任务**再逐个执行(如 Plan-and-Execute、Tree of Thoughts)。
  **Planning:** decompose complex tasks into subtasks first (Plan-and-Execute, Tree of Thoughts).
- **记忆(memory)**：短期(对话/scratchpad) + 长期(把经验存进**向量库**, 用 RAG 检索回来——12.12/12.13 在这里复用!)。
  **Memory:** short-term (conversation/scratchpad) + long-term (store experience in a **vector DB**, retrieve via RAG — reusing 12.12/12.13!).
- **反思(reflection)**：执行后**自我检查**结果对不对, 错了就重试(Reflexion)。
  **Reflection:** self-check results after acting, retry on failure (Reflexion).
- **多智能体(multi-agent)**：多个各有专长的 Agent **协作**(如一个写代码、一个审查、一个测试)。
  **Multi-agent:** several specialized agents **collaborate** (coder, reviewer, tester).

**风险(面试重点)**：① **失控/无限循环**(必须设最大步数、超时——我们的 `max_steps`)；② **工具滥用/安全**(能执行代码/发请求 → 要沙箱、权限控制、人类确认);③ **错误累积**(一步错步步错);④ **成本**(每步一次 LLM 调用, 多步很贵)。让 Agent 可靠仍是开放难题。
**Risks:** ① **runaway/infinite loops** (need max steps/timeouts — our `max_steps`); ② **tool misuse/safety** (executing code/requests → sandbox, permissions, human-in-the-loop); ③ **error compounding** (one wrong step derails the rest); ④ **cost** (an LLM call per step, multi-step is pricey). Making agents reliable is still an open challenge.

```
Agent = LLM大脑 + 工具(搜索/计算/数据库/代码/API) + 自主循环; 让LLM"会思考也会做事"
ReAct循环: Thought(想)→Action(调工具)→Observation(看结果)→重复→Answer; 直到完成
function calling: LLM输出结构化工具调用(JSON), 系统执行后把结果喂回
扩展: 规划(分解子任务)+记忆(短期scratchpad/长期向量库RAG)+反思(自检重试)+多智能体协作
风险: 失控/无限循环(设max_steps) + 工具安全(沙箱/权限/人类确认) + 错误累积 + 成本(每步一次LLM)
应用: AutoGPT/LangChain/AutoGen; 编程助手(Cursor/Claude Code)/客服/RPA
```

### 💡 面试速查 / Interview cheat-sheet
1. **Agent**: LLM大脑 + 工具 + 循环; 会调用工具完成任务(非只生成文本)。
   Agent: LLM brain + tools + loop; calls tools to do tasks, not just generate.
2. **ReAct**: Thought→Action(工具)→Observation→重复→Answer。
   ReAct: Thought→Action(tool)→Observation→repeat→Answer.
3. **function calling**: LLM输出结构化工具调用, 系统执行回填结果。
   Function calling: LLM emits structured tool calls, system runs & returns results.
4. **记忆**: 长期记忆常用向量库+RAG(复用12.12/12.13)。
   Memory: long-term often via vector DB + RAG.
5. **风险**: 失控(设步数上限)/工具安全(沙箱权限)/错误累积/成本。
   Risks: runaway (step limits)/tool safety (sandbox)/error compounding/cost.

### 下一节 / Next
**12.15 LLM 评估**——LLM 生成的文本怎么评好坏? 没有唯一正确答案, 评估很难。我们会讲**困惑度(perplexity)**, 实现 **BLEU/ROUGE** 等文本相似度指标, 并理解 **LLM-as-judge** 等现代评估方法。
**12.15 LLM Evaluation** — how to judge generated text? No single right answer makes evaluation hard. We'll cover **perplexity**, implement **BLEU/ROUGE**, and understand modern methods like **LLM-as-judge**.
